# Agent with Short Term Memory

Memory in AI agents is essential for maintaining continuity, efficiency, and personalization across interactions. Short-term memory, often managed through conversation history within a thread, allows the agent to keep track of ongoing discussions, but long contexts can overwhelm models due to limited context windows, slower responses, and distraction from stale information. To address this, techniques like summarization, windowing, selective recall, and forgetting outdated content help preserve relevance while reducing computational cost. Long-term memory complements this by storing durable facts about user preferences and behaviors across sessions, enabling agents to adapt and provide more satisfying, personalized experiences over time.

In Memory Saver, we can use the `ThreadMemory` class to manage short-term memory by maintaining a conversation history within a thread. This allows the agent to keep track of ongoing discussions and recall relevant information when needed. However, as the conversation history grows, it can become unwieldy and may exceed the context window

In [1]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage,AIMessage

llm = ChatOllama(model="gemma4:e4b")

agent = create_agent(
    llm,
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant that remembers the user's name and age in the current conversation thread."
)

def response_printer(response):
    print("#"*100)
    for message in response["messages"]:
        if isinstance(message, AIMessage):
            print(f"AI: {message.content}")
        elif isinstance(message, HumanMessage):
            print(f"Human: {message.content}")
    print("#"*100)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Bob. I'm 546 years old."}]},
    {"configurable": {"thread_id": "1"}},
)
response_printer(response)


response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name? and how old am I?"}]},
    {"configurable": {"thread_id": "1"}},
)
response_printer(response)

####################################################################################################
Human: Hi! My name is Bob. I'm 546 years old.
AI: Hi Bob! It's nice to meet you. I'll remember that your name is Bob and that you are 546 years old.

How can I help you today?
####################################################################################################
####################################################################################################
Human: Hi! My name is Bob. I'm 546 years old.
AI: Hi Bob! It's nice to meet you. I'll remember that your name is Bob and that you are 546 years old.

How can I help you today?
Human: What is my name? and how old am I?
AI: Your name is Bob, and you are 546 years old.
####################################################################################################


Sqlite State Saver, on the other hand, provides a more structured and persistent way to manage memory. By storing information in a SQLite database, we can efficiently query and retrieve relevant data without overwhelming the model with a long conversation history. This approach allows for better organization of information and can help the agent recall specific details without needing to process the entire conversation history.
Not recommended for production use, but it is a great way to demonstrate how to implement a custom state saver and how to use it with the agent. In this example, we will create a simple SQLite database to store user information such as name and age, and we will use this information to personalize the agent's responses.

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver  
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage,AIMessage
import sqlite3

llm = ChatOllama(model="gemma4:e4b")
conn = sqlite3.connect('memory.db',check_same_thread=False) 

agent = create_agent(
    llm,
    checkpointer=SqliteSaver(conn),
    system_prompt="You are a helpful assistant that remembers the user's name and age in the current conversation thread."
)

def response_printer(response):
    print("#"*100)
    for message in response["messages"]:
        if isinstance(message, AIMessage):
            print(f"AI: {message.content}")
        elif isinstance(message, HumanMessage):
            print(f"Human: {message.content}")
    print("#"*100)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Bob. I'm 546 years old."}]},
    {"configurable": {"thread_id": "2"}},
)
response_printer(response)


response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name? and how old am I?"}]},
    {"configurable": {"thread_id": "2"}},
)
response_printer(response)

####################################################################################################
Human: Hi! My name is Bob. I'm 546 years old.
AI: Hello Bob! It's nice to meet you. I've got your information: you are Bob, and you are 546 years old.

How can I help you today?
####################################################################################################
####################################################################################################
Human: Hi! My name is Bob. I'm 546 years old.
AI: Hello Bob! It's nice to meet you. I've got your information: you are Bob, and you are 546 years old.

How can I help you today?
Human: What is my name? and how old am I?
AI: Your name is Bob, and you are 546 years old.
####################################################################################################
